# Week 5 — Visualization Lab: From Raw File to Insight

A real dataset never arrives ready to plot. This lab walks the full journey an analyst takes: **load the data, clean it, then let the visuals reveal what's inside.** Plotting dirty data produces misleading pictures, so the order matters — clean first, then visualize.

**The dataset.** `car_listings.csv` — used-car adverts from Pakistani classifieds: make, city, year, engine size, fuel type, transmission, mileage and asking price. Like all real data, it is messy: missing values, inconsistent spelling, duplicate adverts, and a few impossible entries.

**The workflow.**

| Stage | Goal |
|---|---|
| 1. Load & inspect | See the size, columns and problems |
| 2. Clean | Fix duplicates, casing, missing values, impossible outliers |
| 3. Feature prep | Derive columns that make analysis easier |
| 4. Visualize — distributions | Understand each column on its own |
| 5. Visualize — categories | Compare groups |
| 6. Visualize — relationships | See what drives price |
| 7. Report | State findings in plain words |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")   # a clean look for every plot in this lab

---
# Stage 1 — Load & inspect

Before touching anything, understand what you have.

In [ ]:
df = pd.read_csv("car_listings.csv")
print("shape:", df.shape)
df.head()

In [ ]:
df.info()

`info()` reveals which columns have missing values — here `mileage_km` and `price_pkr` both have gaps (their non-null counts are below the row total).

In [ ]:
df.describe().round(0)

Scan `describe()` for anything impossible. Look at the **max** of `mileage_km` — a value like 999999 is not a real odometer reading; it's a data-entry error we'll fix in cleaning. This is exactly why we inspect before we plot: one absurd value would stretch a chart's axis and hide everything else.

In [ ]:
df.isnull().sum()

---
# Stage 2 — Clean

Four problems, fixed in order: duplicates, inconsistent text, impossible outliers, then missing values.

### 2.1 Duplicate adverts

In [ ]:
print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("rows now:", len(df))

### 2.2 Inconsistent text

The same make or fuel type has been typed with different capitalization, so the computer sees them as separate categories.

In [ ]:
print("makes before:", sorted(df["make"].unique()))
print("fuels before:", sorted(df["fuel_type"].unique()))

In [ ]:
df["make"]      = df["make"].str.strip().str.title()
df["fuel_type"]  = df["fuel_type"].str.strip().str.title()

print("makes after:", sorted(df["make"].unique()))
print("fuels after:", sorted(df["fuel_type"].unique()))

Now each make and fuel type is a single category. Without this step, a plot of "average price by make" would split Toyota into `Toyota` and `TOYOTA` and mislead completely.

### 2.3 Impossible outliers

The odometer placeholder `999999` is not a real reading. Treat it as missing so it can be handled with the genuine gaps.

In [ ]:
print("suspicious mileage rows:", (df["mileage_km"] == 999999).sum())
df["mileage_km"] = df["mileage_km"].replace(999999, np.nan)

### 2.4 Missing values

Both numeric columns are skewed (a few very expensive cars, a few very high-mileage cars), so the **median** is the safe fill value — it isn't dragged around by the extremes.

In [ ]:
for col in ["mileage_km", "price_pkr"]:
    print(f"{col:12s} mean={df[col].mean():12,.0f}   median={df[col].median():12,.0f}   skew={df[col].skew():.2f}")

In [ ]:
df["mileage_km"] = df["mileage_km"].fillna(df["mileage_km"].median())
df["price_pkr"]   = df["price_pkr"].fillna(df["price_pkr"].median())

print("remaining missing values:", df.isnull().sum().sum())

---
# Stage 3 — Feature preparation

Derive a couple of columns that make the analysis more natural. A car's **age** is easier to reason about than its model year, and a **price in lakhs** is easier to read than a long rupee figure.

In [ ]:
df["age"]          = 2024 - df["year"]
df["price_lakh"]    = df["price_pkr"] / 100_000     # 1 lakh = 100,000
df[["year", "age", "price_pkr", "price_lakh"]].head()

We can also bin mileage into readable bands — turning a continuous number into an ordered category, the way we did last week.

In [ ]:
df["mileage_band"] = pd.cut(df["mileage_km"],
                            bins=[0, 50_000, 100_000, 150_000, 1_000_000],
                            labels=["low", "medium", "high", "very high"])
df["mileage_band"].value_counts().sort_index()

---
# Stage 4 — Visualize: distributions

Now the data is trustworthy. Start by understanding each important column on its own.

In [ ]:
# Price distribution
sns.histplot(data=df, x="price_lakh", kde=True, bins=25)
plt.title("Distribution of car prices")
plt.xlabel("price (lakh PKR)")
plt.show()

Mark the mean and median to reveal the skew, just as in the lesson.

In [ ]:
sns.histplot(data=df, x="price_lakh", kde=True, bins=25)
plt.axvline(df["price_lakh"].mean(),   color="red",   linestyle="--", label="mean")
plt.axvline(df["price_lakh"].median(), color="green", linestyle="--", label="median")
plt.title("Car prices — mean vs median")
plt.xlabel("price (lakh PKR)")
plt.legend()
plt.show()

The mean sits to the right of the median: a right-skewed distribution, driven by a minority of expensive cars. That skew is why we filled the missing prices with the median.

In [ ]:
# Mileage distribution as a boxplot — the box, whiskers and any outlier dots
sns.boxplot(data=df, x="mileage_km")
plt.title("Mileage distribution (box plot)")
plt.xlabel("mileage (km)")
plt.show()

---
# Stage 5 — Visualize: comparing categories

How does price differ across the categorical columns — make, fuel, transmission?

In [ ]:
# How many listings per make?
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="make", order=df["make"].value_counts().index)
plt.title("Number of listings by make")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Average price by make
plt.figure(figsize=(8, 4))
sns.barplot(data=df, x="make", y="price_lakh",
            order=df.groupby("make")["price_lakh"].mean().sort_values(ascending=False).index)
plt.title("Average price by make")
plt.ylabel("mean price (lakh PKR)")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Price spread by fuel type — a boxplot shows more than a bar
sns.boxplot(data=df, x="fuel_type", y="price_lakh")
plt.title("Price by fuel type")
plt.ylabel("price (lakh PKR)")
plt.show()

In [ ]:
# Does transmission relate to price?
sns.violinplot(data=df, x="transmission", y="price_lakh")
plt.title("Price by transmission (violin plot)")
plt.ylabel("price (lakh PKR)")
plt.show()

A violin plot is a boxplot that also shows the *shape* of each group's distribution — wider sections mean more cars at that price.

---
# Stage 6 — Visualize: relationships

The central question for a car dataset: **what drives the price?** Relationships answer it.

In [ ]:
# Age vs price
sns.regplot(data=df, x="age", y="price_lakh",
            scatter_kws={"alpha": 0.4}, line_kws={"color": "red"})
plt.title("Older cars are cheaper")
plt.xlabel("age (years)")
plt.ylabel("price (lakh PKR)")
plt.show()

A clear downward trend: the older the car, the lower the price. The red line's negative slope is the relationship, and it matches intuition.

In [ ]:
# Mileage vs price
sns.regplot(data=df, x="mileage_km", y="price_lakh",
            scatter_kws={"alpha": 0.4}, line_kws={"color": "red"})
plt.title("Higher mileage, lower price")
plt.xlabel("mileage (km)")
plt.ylabel("price (lakh PKR)")
plt.show()

In [ ]:
# The whole picture at once: a correlation heatmap of the numeric columns
numeric = ["age", "engine_cc", "mileage_km", "price_lakh"]
sns.heatmap(df[numeric].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("What relates to price?")
plt.show()

Read the `price_lakh` row: **age** and **mileage** are strongly *negative* (cool colours — as they rise, price falls), while **engine size** is mildly positive. One picture summarizes every numeric relationship in the dataset.

In [ ]:
# Colour a scatter by a category to see two effects at once
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="price_lakh", hue="fuel_type", alpha=0.7)
plt.title("Age vs price, split by fuel type")
plt.xlabel("age (years)")
plt.ylabel("price (lakh PKR)")
plt.show()

---
# Stage 7 — Report

Gather the headline numbers, then state the findings in plain language.

In [ ]:
print("CLEAN DATASET")
print(f"  listings: {len(df)}   missing values: {df.isnull().sum().sum()}")
print()
print("PRICE (lakh PKR)")
print(f"  mean {df['price_lakh'].mean():.1f}   median {df['price_lakh'].median():.1f}   max {df['price_lakh'].max():.1f}")
print()
print("AVERAGE PRICE BY FUEL TYPE")
print(df.groupby('fuel_type')['price_lakh'].mean().round(1).to_string())
print()
print("WHAT CORRELATES WITH PRICE")
print(df[numeric].corr()['price_lakh'].drop('price_lakh').round(2).to_string())

### Findings

*(Write 3–5 plain-English bullet points, each backed by a plot or a number above. Example: "A car's age is the strongest driver of price — older cars are consistently cheaper, shown by the downward trend and a correlation of about −X.")*

1. 
2. 
3. 

---
## What this lab demonstrated

Starting from a raw, messy CSV, we:

- **inspected** the data and spotted an impossible mileage value before it could distort any chart,
- **cleaned** duplicates, inconsistent spelling, the impossible outlier, and missing values,
- **prepared** friendlier columns (age, price in lakhs, mileage bands),
- **visualized distributions** to understand each column and see the price skew,
- **compared categories** to see how make, fuel and transmission relate to price,
- **explored relationships** to find what actually drives price — and summarized them all in one heatmap,
- **reported** the findings in words anyone could follow.

The plots did not create the insight — the **cleaning did**. A beautiful chart built on dirty data is a confident lie. Clean first, visualize second, and always say what you found in plain words.